<a id="setup"></a>
# <p style="background-color: #ff6200; font-family:calibri; color:white; font-size:140%; font-family:Verdana; text-align:center; border-radius:15px 50px;">Chapter 3 | Filter and Join Queries</p>

In [4]:
import importlib
import lab2 as lab2


# Réimporter les classes
from lab2 import Database, Collection

<a id="libraries"></a>
# <b><span style='color:#fcc36d'>0|</span><span style='color:#ff6200'> Mock Databases construction </span></b>

##### Sub-bloc of the Database

In [5]:
# Sub blocs Category, Supplier, etc...
# Category
schema_category = {
    "type": "object",
    "properties": {"title": {"type": "string"}},
    "required": ["title"]
}

# Supplier
schema_supplier = {
    "type": "object",
    "properties": {
        "IDS": {"type": "integer"},
        "name": {"type": "string"},
        "SIRET": {"type": "string"},
        "headOffice": {"type": "string"},
        "Revenue": {"type": "integer"}
    },
    "required": ["IDS", "name", "SIRET", "headOffice", "Revenue"]
}

# Price
schema_price = {
    "type": "object",
    "properties": {
        "amount": {"type": "number"},
        "currency": {"type": "string"},
        "VAT": {"type": "number"}
    },
    "required": ["amount", "currency", "VAT"]
}

# Product (Categories + Suppliers embedded)
product_embedded_schema = {
    "type": "object",
    "properties": {
        "IDP": {"type": "integer"},
        "name": {"type": "string"},
        "brand": {"type": "string"},
        "description": {"type": "string"},
        "image_url": {"type": "string"},
        "price": schema_price,
        # Nesting [Cat]
        "categories": {"type": "array", "items": schema_category}, 
        # Nesting Supp
        "supplier": schema_supplier                               
    },
    "required": ["IDP", "name", "brand", "description", "image_url", "price", "categories", "supplier"]
}

# Warehouse 
warehouse_schema = {
    "type": "object", 
    "properties": {
        "IDW": {"type": "integer"}, 
        "address": {"type": "string"},
        "capacity": {"type": "integer"} 
    },
    "required": ["IDW", "address", "capacity"]
}

# Stock
stock_schema = {
    "type": "object", 
    "properties": {
        "IDP": {"type": "integer"}, 
        "IDW": {"type": "integer"}, 
        "quantity": {"type": "integer"},
        "location": {"type": "string"}
    },
    "required": ["IDP", "IDW", "quantity", "location"]    
}

# Order Line
order_line_schema = {
    "type": "object", 
    "properties": {
        "IDP": {"type": "integer"},
        "IDC": {"type": "integer"},
        "date": {"type": "string"},
        "quantity": {"type": "integer"},
        "deliveryDate": {"type": "string"},
        "comment": {"type": "string"},
        "grade": {"type": "integer"}
    },
    "required": ["IDP", "IDC", "quantity", "date", "deliveryDate", "comment", "grade"]
}

# Client
client_schema = {
    "type": "object", 
    "properties": {
        "IDC": {"type": "integer"},
        "ln": {"type": "string"},
        "fn": {"type": "string"},   
        "address": {"type": "string"},
        "nationality": {"type": "string"},
        "birthDate": {"type": "string"},
        "email": {"type": "string"}
    },
    "required": ["IDC", "ln", "fn", "address", "nationality", "birthDate", "email"]
}

#### Mock DBs Schema

In [6]:
# Stats 

STATS = {
    "nb_products": 10**5,        # 100,000 Products
    "nb_clients": 10**7,         # 10 Million Clients
    "nb_warehouses": 200,        # 200 Warehouses
    "nb_orderlines": 4 * 10**9,  # 4 Billion Order Lines
    "nb_brands": 5000,
    # Derived stats for array multipliers
    "avg_cat_per_prod": 2        # Average categories per product
}

In [7]:
# Mock DB1

db1_schemas = {
    "Product": {
        "type": "object",
        "properties": {
            "IDP": {"type": "integer"},
            "name": {"type": "string"},
            "brand": {"type": "string"},
            "description": {"type": "string"}, 
            "image_url": {"type": "string"},   
            "price": schema_price,
            
            # Nesting: Array of Categories
            "categories": {"type": "array", "items": schema_category},
            
            # Nesting: Supplier Object
            "supplier": schema_supplier 
        },
        "required": ["IDP", "name", "brand", "description", "image_url", "price", "categories", "supplier"]
    },
    
    "Stock": stock_schema,
    "Warehouse": warehouse_schema,
    "OrderLine": order_line_schema,
    "Client": client_schema
}

db1 = Database("DB1 - Normalized")

# Create Collections
c_prod = Collection("Product", db1_schemas["Product"], stats=STATS)
c_stock = Collection("Stock", db1_schemas["Stock"], stats=STATS, count_rule="product_x_warehouse")
c_warehouse = Collection("Warehouse", db1_schemas["Warehouse"], stats=STATS)
c_orderline = Collection("OrderLine", db1_schemas["OrderLine"], stats=STATS)
c_client = Collection("Client", db1_schemas["Client"], stats=STATS)

# Add to DB
db1.add_collection(c_prod)
db1.add_collection(c_stock)
db1.add_collection(c_warehouse)
db1.add_collection(c_orderline)
db1.add_collection(c_client)

In [8]:
# Mock DB2

db4_schemas = {
    "OrderLine": {
        "type": "object",
        "properties": {
            "IDC": {"type": "integer"},
            "date": {"type": "string"},
            "quantity": {"type": "integer"},
            "deliveryDate": {"type": "string"},
            "comment": {"type": "string"},
            "grade": {"type": "integer"},
            
            # Nesting: Product + Categories + Supplier
            "product": product_embedded_schema
        },
        "required": ["IDC", "date", "quantity", "grade", "product"]
    },
    "Stock": stock_schema,
    "Warehouse": warehouse_schema,
    "Client": client_schema
}

db4 = Database("DB4 - Denormalized")

# Create Collections
c_orderline_embedded = Collection("OrderLine", db4_schemas["OrderLine"], stats=STATS)
c_stock_db4 = Collection("Stock", db4_schemas["Stock"], stats=STATS, count_rule="product_x_warehouse")
c_warehouse_db4 = Collection("Warehouse", db4_schemas["Warehouse"], stats=STATS)
c_client_db4 = Collection("Client", db4_schemas["Client"], stats=STATS)

# Add to DB
db4.add_collection(c_orderline_embedded)
db4.add_collection(c_stock_db4)
db4.add_collection(c_warehouse_db4)
db4.add_collection(c_client_db4)
